# 35 — exit=0.0 Rebuild

Two purposes:

**Part 1 — exit=0.0 rebuild:** The current backtest uses a stateless direction column — position
is active only while |z| > 2.0, so it exits the moment |z| drops below entry. The roadmap (notebook 22)
and the source docstring (`richness.py:131`) both specify *"z → 0 → exit"*, but `build_signal_panel`
has no `exit_z` parameter and never implemented that logic. This notebook:
- Documents the evidence
- Applies exit=0.0 hysteresis (hold until z crosses zero)
- Rebuilds all positions and PnL
- Reports new vs old headline numbers and regime/stress/IEF analyses side-by-side

**Part 2 — hedge weight spot-check:** Computes actual hedge weight magnitudes for the proposed
[2Y, 10Y, 20Y] hedge set at 9 dates spanning 1990–2024, then flags the 20Y liquidity caveat.

Old values for comparison come from:
- Notebook 29 (tearsheet): gross Sharpe 0.29, net Sharpe 0.11, net P&L \$313,804, net DD -\$647,353
- Notebook 30 (regime): all-month Sharpe 0.12
- Notebook 31 (stress): PC1 net -\$11,882 ; 2013 Taper historical net -\$77,621
- Notebook 33 (IEF beta): beta = -\$71/1%, p = 0.58

## Exploration results — both proposals rejected. Production baseline unchanged.

**Exit-band hysteresis (exit at z=0 instead of |z|<2.0):** Raised net Sharpe from 0.11 to 0.32
and net P&L from $314K to $1.53M, but was **rejected** because IEF beta became statistically
significant: +$1,042/1% IEF move (t=2.17, p=0.030) versus −$71 (p=0.58) at baseline.
The longer average holding period (18.3 days vs 4.4 days) breaks factor neutrality —
hedge weights are solved once at trade entry and never updated while the position is open,
so a position held through a full z=0 reversion accumulates unhedged duration drift.

A production version of exit-band hysteresis would require **periodic hedge re-solving
during open positions** (e.g. daily or weekly recalibration of the 3×3 KRD system),
followed by re-validation via IEF beta, before it could be trusted. Documented as future work.

**Universe expansion to [2Y, 10Y, 20Y] hedges:** Also **rejected**. Pre-2020 20Y Treasuries
were discontinued/illiquid; the flat 0.02 bp/leg cost model understates their real execution
cost by 10–100×, so the well-conditioned matrix result (condition# 8–11 vs 95–122 for current
[2Y, 5Y, 10Y]) does not translate to a realistic backtest. The 20Y weight is also negligibly
small (< 0.04× signal face), confirming the 5Y already does most of the work.

**Current production baseline (all committed Week 7 results):**
exit threshold = 2.0 (stateless) | hedge instruments = [2Y, 5Y, 10Y] | gross Sharpe 0.29 | net Sharpe 0.11

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import inspect

from termstructure.signals import richness
from termstructure.backtest.pnl import compute_net_pnl
from termstructure.backtest.portfolio import build_portfolio_history
from termstructure.report.tearsheet import (
    print_stats_table, DEFAULT_COST_BPS, DEFAULT_POS_SCALE,
)
from termstructure.risk.hedge import build_hedge
from termstructure.curves.svensson import svensson_zero_rate

_ROOT = Path('..').resolve()
_DATA = _ROOT / 'data' / 'processed'
_FINE_GRID = np.array([0.5, 1, 2, 3, 5, 7, 10, 15, 20, 25, 30], dtype=float)

# === Evidence of the unintentional exit=2.0 deviation ===
print('Evidence that exit=2.0 (stateless) was unintentional:\n')
print('1. richness.py compute_zscore_signal docstring (the function that produces z-scores):')
print('      z > +2.0  ->  cheap  (buy)')
print('      z < -2.0  ->  rich   (sell)')
print('      |z| < 2.0 ->  no signal')
print('      z -> 0    ->  exit        <-- documented spec')
print()
print('2. build_signal_panel function signature:')
src = inspect.getsource(richness.build_signal_panel)
# Print first 8 lines (signature + docstring start)
for line in src.split('\n')[:12]:
    print(f'   {line}')
print()
print('3. Notebook 22 header table: "z -> 0: back to average: exit"')
print()
print('=> build_signal_panel only takes entry_z. It assigns direction=+1 when z>2.0,')
print('   direction=0 when |z|<=2.0. No state machine, no exit_z parameter.')
print('   Result: position turns off the moment |z| drops below 2.0 -- not at zero-crossing.')
print('   This is exit=2.0 behavior, not the exit=0.0 the roadmap specified.')

Evidence that exit=2.0 (stateless) was unintentional:

1. richness.py compute_zscore_signal docstring (the function that produces z-scores):
      z > +2.0  ->  cheap  (buy)
      z < -2.0  ->  rich   (sell)
      |z| < 2.0 ->  no signal
      z -> 0    ->  exit        <-- documented spec

2. build_signal_panel function signature:
   def build_signal_panel(entry_z: float = 2.0) -> pd.DataFrame:
       """
       Build the final backtest-ready signal panel from the z-score signal.
   
       Adds a `direction` column encoding the trade direction at the given entry
       threshold, then saves the full panel (all dates, all maturities, including
       direction=0 rows) so Week 6 can query any date without re-implementing
       threshold logic.
   
       direction encoding:
           +1  z >  +entry_z   cheap → buy
           -1  z <  -entry_z   rich  → sell

3. Notebook 22 header table: "z -> 0: back to average: exit"

=> build_signal_panel only takes entry_z. It assigns direction=+1

## Part 1 — Rebuild with exit=0.0 hysteresis

In [2]:
# Load z-scores (no direction), back up old signal panel, apply hysteresis
sig = pd.read_parquet(_DATA / 'richness_signal.parquet')
sig['date'] = pd.to_datetime(sig['date'])

old_sp = pd.read_parquet(_DATA / 'signal_panel.parquet')
old_sp.to_parquet(_DATA / 'signal_panel_stateless.parquet', index=False)
print('Backed up old signal panel -> signal_panel_stateless.parquet')


def apply_hysteresis(sig_df, entry_z=2.0, exit_z=0.0):
    """Stateful entry/exit: enter at |z|>=entry_z, exit when z crosses exit_z toward zero."""
    result = []
    for mat, grp in sig_df.groupby('maturity'):
        grp = grp.sort_values('date').copy()
        state = 0
        dirs = []
        for z in grp['z_score']:
            if state == 0:
                if z >= entry_z:    state = 1
                elif z <= -entry_z: state = -1
            elif state == 1:
                if z < exit_z:  state = 0   # exit long when z drops below 0
            else:                            # state == -1
                if z > -exit_z: state = 0   # exit short when z rises above 0
            dirs.append(state)
        grp['direction'] = dirs
        result.append(grp)
    return pd.concat(result, ignore_index=True).sort_values(['date', 'maturity']).reset_index(drop=True)


new_sp = apply_hysteresis(sig, entry_z=2.0, exit_z=0.0)
new_sp.to_parquet(_DATA / 'signal_panel.parquet', index=False)
print('Saved new signal_panel.parquet (exit=0.0 hysteresis)\n')

# Coverage comparison for signal maturities only
print('Signal coverage change (3Y and 7Y only -- the two signal maturities):')
print(f'  {"Mat":<5} {"Old active":>12} {"New active":>12} {"Delta":>10} {"Old %":>8} {"New %":>8}')
print(f'  {"-"*57}')
for mat in [3, 7]:
    total    = (old_sp['maturity'] == mat).sum()
    old_act  = (old_sp[old_sp['maturity'] == mat]['direction'] != 0).sum()
    new_act  = (new_sp[new_sp['maturity'] == mat]['direction'] != 0).sum()
    print(f'  {mat}Y    {old_act:>12,} {new_act:>12,} {new_act-old_act:>+10,} '
          f'{100*old_act/total:>7.1f}% {100*new_act/total:>7.1f}%')

Backed up old signal panel -> signal_panel_stateless.parquet
Saved new signal_panel.parquet (exit=0.0 hysteresis)

Signal coverage change (3Y and 7Y only -- the two signal maturities):
  Mat     Old active   New active      Delta    Old %    New %
  ---------------------------------------------------------
  3Y           4,205        4,205         +0    31.6%    31.6%
  7Y           4,293        4,293         +0    32.3%    32.3%


In [3]:
# Rebuild portfolio positions
# Reads the new signal_panel.parquet, overwrites portfolio_positions.parquet
# Runtime: ~3-8 minutes (iterates over ~14k dates)
old_pos = pd.read_parquet(_DATA / 'portfolio_positions.parquet')
old_pos.to_parquet(_DATA / 'portfolio_positions_stateless.parquet', index=False)
print(f'Backed up {len(old_pos):,} old rows ({old_pos["date"].nunique():,} dates) '
      f'-> portfolio_positions_stateless.parquet')
print('Rebuilding positions...')

new_pos = build_portfolio_history()  # writes portfolio_positions.parquet

print(f'\n  Old: {len(old_pos):,} rows  |  {old_pos["date"].nunique():,} active dates')
print(f'  New: {len(new_pos):,} rows  |  {new_pos["date"].nunique():,} active dates')

Backed up 33,992 old rows (5,466 dates) -> portfolio_positions_stateless.parquet
Rebuilding positions...
Saved 33,992 portfolio rows -> C:\Users\aarna\Documents\termstructure\data\processed\portfolio_positions.parquet

  Old: 33,992 rows  |  5,466 active dates
  New: 33,992 rows  |  5,466 active dates


## 1. Full PnL comparison

In [4]:
print('=' * 80)
print('NEW -- exit=0.0 hysteresis (hold until z crosses zero)')
print('=' * 80)
print_stats_table(cost_bps=DEFAULT_COST_BPS, position_scale=DEFAULT_POS_SCALE)

print('\n' + '=' * 80)
print('OLD -- exit=2.0 stateless (notebook 29 reference)')
print('=' * 80)
print("""
  Factor-neutral RV strategy -- backtest tearsheet
  Cost assumption: 0.02bp per leg per event  |  Position scale: 0.8x
---------------------------------------------------------------------------------------------------
Metric                                          Gross                 Scaled                    Net
---------------------------------------------------------------------------------------------------
  Total P&L ($)                            $1,009,244               $807,396               $313,804
  Ann. avg P&L ($)                            $19,068                $15,255                 $5,929
  Sharpe                                         0.29                   0.29                   0.11
  Max drawdown ($)                          $-558,322              $-446,657              $-647,353
  Calmar ratio                                   0.03                   0.03                   0.01
  Hit rate                                      50.0%                  50.0%                  47.4%
  Active days                                   1,305                  1,305                  1,305
---------------------------------------------------------------------------------------------------
  Backtest span (years)                          52.9
  Round trips (total)                             834
  Round trips / year                             15.8
  Avg holding (days)                              4.4
  Total transaction cost ($)                 $493,592
""")

NEW -- exit=0.0 hysteresis (hold until z crosses zero)
Saved 5,466 daily P&L rows -> C:\Users\aarna\Documents\termstructure\data\processed\portfolio_pnl.parquet

  Factor-neutral RV strategy -- backtest tearsheet
  Cost assumption: 0.02bp per leg per event  |  Position scale: 0.8x
---------------------------------------------------------------------------------------------------
Metric                                          Gross                 Scaled                    Net
---------------------------------------------------------------------------------------------------
  Total P&L ($)                            $2,388,464             $1,910,771             $1,528,831
  Ann. avg P&L ($)                            $45,087                $36,070                $28,860
  Sharpe                                         0.40                   0.40                   0.32
  Max drawdown ($)                          $-858,329              $-686,663              $-716,138
  Calmar ratio    

## 2. Regime analysis (exit=0.0 vs exit=2.0)

In [5]:
fs = pd.read_parquet(_DATA / 'factor_scores.parquet')
fs['date'] = pd.to_datetime(fs['date'])

net = compute_net_pnl(position_scale=DEFAULT_POS_SCALE, cost_bps=DEFAULT_COST_BPS)
net['date'] = pd.to_datetime(net['date'])

df = fs[['date', 'score_1', 'score_2']].merge(net[['date', 'net_pnl']], on='date', how='left')
df['net_pnl'] = df['net_pnl'].fillna(0.0)
df = df[df['date'] <= net['date'].max()].copy().sort_values('date').reset_index(drop=True)
df['pc2_level'] = df['score_2'].cumsum()

monthly = (
    df.set_index('date').resample('ME')
    .agg(
        pc1_vol   = ('score_1', 'std'),
        pc1_shift = ('score_1', 'sum'),
        pc2_level = ('pc2_level', 'mean'),
        net_pnl   = ('net_pnl', 'sum'),
        n_days    = ('net_pnl', 'count'),
    )
    .dropna(subset=['pc1_vol']).reset_index()
)
monthly = monthly[monthly['n_days'] >= 10].copy()
monthly['vol_regime'], vol_bins = pd.qcut(
    monthly['pc1_vol'], 3, labels=['Low', 'Med', 'High'], retbins=True
)
monthly['rate_dir']    = (monthly['pc1_shift'] > 0).map({True: 'Rising', False: 'Falling'})
monthly['curve_shape'] = (monthly['pc2_level'] > monthly['pc2_level'].median()).map(
    {True: 'Steep', False: 'Flat'}
)


def monthly_sharpe(s):
    return np.nan if len(s) < 6 else s.mean() / s.std() * np.sqrt(12)


# Old Sharpe values from notebook 30
OLD_REGIME = {
    'vol':   {'Low': 0.10, 'Med': 0.27, 'High': 0.02},
    'rate':  {'Falling': 0.28, 'Rising': -0.03},
    'curve': {'Steep': -0.02, 'Flat': 0.27},
}

dims = [
    ('Volatility regime (PC1 realized vol)', 'vol_regime',  ['Low', 'Med', 'High'], 'vol'),
    ('Rate direction (monthly PC1 shift)',   'rate_dir',    ['Falling', 'Rising'],  'rate'),
    ('Curve shape (cumulative PC2 level)',   'curve_shape', ['Steep', 'Flat'],      'curve'),
]

for title, col, labels, key in dims:
    print(f'\n{title}')
    print(f'  {"Label":<10} {"Months":>6} {"New Net P&L":>13} {"New Sharpe":>11} {"Old Sharpe":>11}')
    print(f'  {"-" * 55}')
    for label in labels:
        sub = monthly.loc[monthly[col] == label, 'net_pnl']
        new_sh = monthly_sharpe(sub)
        old_sh = OLD_REGIME[key][label]
        new_str = f'{new_sh:.2f}' if not np.isnan(new_sh) else '  n/a'
        print(f'  {label:<10} {len(sub):>6} ${sub.sum():>11,.0f} {new_str:>11} {old_sh:>11.2f}')
    all_pnl = monthly['net_pnl']
    new_all = monthly_sharpe(all_pnl)
    print(f'  {"ALL":<10} {len(monthly):>6} ${all_pnl.sum():>11,.0f} {new_all:>11.2f} {0.12:>11.2f}')

Saved 5,466 daily P&L rows -> C:\Users\aarna\Documents\termstructure\data\processed\portfolio_pnl.parquet

Volatility regime (PC1 realized vol)
  Label      Months   New Net P&L  New Sharpe  Old Sharpe
  -------------------------------------------------------
  Low           156 $    436,399        0.39        0.10
  Med           155 $    142,428        0.10        0.27
  High          155 $    962,483        0.48        0.02
  ALL           466 $  1,541,310        0.33        0.12

Rate direction (monthly PC1 shift)
  Label      Months   New Net P&L  New Sharpe  Old Sharpe
  -------------------------------------------------------
  Falling       245 $    758,257        0.31        0.28
  Rising        221 $    783,053        0.35       -0.03
  ALL           466 $  1,541,310        0.33        0.12

Curve shape (cumulative PC2 level)
  Label      Months   New Net P&L  New Sharpe  Old Sharpe
  -------------------------------------------------------
  Steep         233 $    335,646     

## 3. Stressed periods comparison

In [6]:
# Old stressed-period net P&L from notebook 30
STRESSED_OLD = [
    ('2013 Taper Tantrum', '2013-05-01', '2013-09-30',  78_411),
    ('March 2020 COVID',   '2020-02-01', '2020-04-30',  15_801),
    ('2022 Inflation',     '2022-01-01', '2022-12-31',  73_606),
]

print(f'{"Period":<25} {"Mos":>4} {"New Net P&L":>14} {"Old Net P&L":>14} {"Delta":>12}')
print('-' * 72)
for name, start, end, old_pnl in STRESSED_OLD:
    mask = (monthly['date'] >= start) & (monthly['date'] <= end)
    sub  = monthly[mask]
    new_pnl = sub['net_pnl'].sum()
    print(f'{name:<25} {len(sub):>4} ${new_pnl:>12,.0f} ${old_pnl:>12,.0f} ${new_pnl-old_pnl:>+10,.0f}')

# Full backtest
print(f'{"Full backtest (1985-2024)":<25} {len(monthly):>4} '
      f'${monthly["net_pnl"].sum():>12,.0f} ${313804:>12,} '
      f'${monthly["net_pnl"].sum()-313804:>+10,.0f}')

Period                     Mos    New Net P&L    Old Net P&L        Delta
------------------------------------------------------------------------
2013 Taper Tantrum           5 $     -43,522 $      78,411 $  -121,933
March 2020 COVID             3 $      48,404 $      15,801 $   +32,603
2022 Inflation              12 $     315,857 $      73,606 $  +242,251
Full backtest (1985-2024)  466 $   1,541,310 $     313,804 $+1,227,506


## 4. Stress scenarios comparison (instantaneous shocks)

In [7]:
loadings_df = pd.read_parquet(_DATA / 'pca_loadings.parquet')
zp = pd.read_parquet(_DATA / 'zero_panel.parquet')
zp['date'] = pd.to_datetime(zp['date'])

positions = pd.read_parquet(_DATA / 'portfolio_positions.parquet')
positions['date'] = pd.to_datetime(positions['date'])
port_mats = [2, 3, 5, 7, 10]  # signal: 3,7 + hedge: 2,5,10

FACTOR_SHOCKS = {'PC1 +100bp': (1, +100), 'PC2 +50bp': (2, +50), 'PC3 +25bp': (3, +25)}
EPISODES      = {
    '2013 Taper Tantrum': ('2013-05-01', '2013-09-30'),
    '2008 September':     ('2008-09-01', '2008-09-30'),
}

# Old net P&L from notebook 31
OLD_STRESS = {
    'PC1 +100bp': -11_882, 'PC2 +50bp': 21_672, 'PC3 +25bp': 820,
    '2013 Taper Tantrum': -77_621, '2008 September': -22_797,
}


def factor_shock_dy(pc, delta):
    row = loadings_df[loadings_df['pc'] == pc].iloc[0]
    return {m: row[f'y_{m}'] * delta for m in port_mats}


def get_snapshot(pos, before_date):
    valid = pos[pos['date'] <= pd.Timestamp(before_date)]
    if valid.empty: return pd.DataFrame(), None
    snap_date = valid['date'].max()
    return pos[pos['date'] == snap_date].copy(), snap_date


def compute_stress_pnl(snap, dy_by_mat):
    rows = []
    for _, row in snap.iterrows():
        m = int(row['leg_maturity'])
        pnl = -np.sign(row['notional']) * row['dv01'] * dy_by_mat.get(m, 0.0)
        rows.append({'leg_type': row['leg_type'], 'pnl': pnl})
    return pd.DataFrame(rows)


def summarise(leg_df):
    return {
        'signal_pnl': leg_df[leg_df['leg_type'] == 'signal']['pnl'].sum(),
        'hedge_pnl':  leg_df[leg_df['leg_type'] == 'hedge']['pnl'].sum(),
        'net_pnl':    leg_df['pnl'].sum(),
    }


snap_latest, snap_date_latest = get_snapshot(positions, '2024-09-30')
print(f'Factor-shock snapshot: {snap_date_latest.date()}\n')
print(f'{"Scenario":<24} {"Signal":>12} {"Hedge":>12} {"New Net":>12} {"Old Net":>12}')
print('=' * 76)

print('--- Factor shocks ---')
for name, (pc, delta) in FACTOR_SHOCKS.items():
    s = summarise(compute_stress_pnl(snap_latest, factor_shock_dy(pc, delta)))
    old = OLD_STRESS[name]
    print(f'{name:<24} ${s["signal_pnl"]:>10,.0f} ${s["hedge_pnl"]:>10,.0f} '
          f'${s["net_pnl"]:>10,.0f} ${old:>10,.0f}')

print('--- Historical replays ---')
for name, (start, end) in EPISODES.items():
    dy = {m: zp[zp['date'].between(start, end)][f'dy_{m}'].sum() for m in port_mats}
    snap, snap_date = get_snapshot(positions, start)
    s = summarise(compute_stress_pnl(snap, dy))
    old = OLD_STRESS[name]
    print(f'{name:<24} ${s["signal_pnl"]:>10,.0f} ${s["hedge_pnl"]:>10,.0f} '
          f'${s["net_pnl"]:>10,.0f} ${old:>10,.0f}')

Factor-shock snapshot: 2024-09-30

Scenario                       Signal        Hedge      New Net      Old Net
--- Factor shocks ---
PC1 +100bp               $    28,662 $   -25,139 $     3,523 $   -11,882
PC2 +50bp                $   178,387 $  -185,739 $    -7,351 $    21,672
PC3 +25bp                $   -86,752 $    90,104 $     3,351 $       820
--- Historical replays ---
2013 Taper Tantrum       $  -600,700 $   447,513 $  -153,187 $   -77,621
2008 September           $  -290,300 $   263,458 $   -26,843 $   -22,797


## 5. IEF beta comparison

In [8]:
try:
    import yfinance as yf
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'yfinance', '-q'])
    import yfinance as yf

from scipy import stats

# New net P&L (already computed from rebuilt positions)
net = compute_net_pnl(position_scale=DEFAULT_POS_SCALE, cost_bps=DEFAULT_COST_BPS)
net['date'] = pd.to_datetime(net['date'])

# IEF daily returns (inception 2002)
ief_raw = yf.download('IEF', start='2002-01-01', end='2025-01-01',
                      auto_adjust=True, progress=False)
# yfinance 1.0+ may return MultiIndex columns for single-ticker downloads
if isinstance(ief_raw.columns, pd.MultiIndex):
    ief_raw.columns = ief_raw.columns.get_level_values(0)
ief = ief_raw[['Close']].copy()
ief.index = pd.to_datetime(ief.index).tz_localize(None)
ief['ief_ret'] = ief['Close'].pct_change() * 100  # %
ief = ief.dropna().reset_index()
ief = ief.rename(columns={ief.columns[0]: 'date'})  # first col is always date after reset_index

merged = net.merge(ief[['date', 'ief_ret']], on='date', how='inner')
slope, intercept, r, p, se = stats.linregress(merged['ief_ret'], merged['net_pnl'])

print(f'New (exit=0.0) IEF beta  [{merged["date"].min().date()} to {merged["date"].max().date()}]')
print(f'  Observations  : {len(merged):,}')
print(f'  Beta          : ${slope:,.0f} / 1% IEF move')
print(f'  t-stat        : {slope/se:.2f}')
print(f'  p-value       : {p:.3f}')
print(f'  R-squared     : {r**2:.5f}')
print()
print('Old (exit=2.0) IEF beta -- notebook 33:')
print('  Beta          : $-71 / 1% IEF move')
print('  t-stat        : -0.55')
print('  p-value       : 0.580')
print('  R-squared     : 0.00005')
print()
if p > 0.05:
    print(f'Both specifications have non-significant IEF beta (new p={p:.3f}).')
    print('Factor-neutral hedge works under both exit rules.')
else:
    print(f'WARNING: new IEF beta is significant (p={p:.3f}). Investigate hedge performance.')

Saved 5,466 daily P&L rows -> C:\Users\aarna\Documents\termstructure\data\processed\portfolio_pnl.parquet
New (exit=0.0) IEF beta  [2002-07-31 to 2024-10-03]
  Observations  : 2,369
  Beta          : $1,041 / 1% IEF move
  t-stat        : 2.17
  p-value       : 0.030
  R-squared     : 0.00198

Old (exit=2.0) IEF beta -- notebook 33:
  Beta          : $-71 / 1% IEF move
  t-stat        : -0.55
  p-value       : 0.580
  R-squared     : 0.00005



## Part 2 — Hedge weight spot-check: proposed [2Y, 10Y, 20Y]

In [9]:
params = pd.read_parquet(_DATA / 'svensson_params.parquet')
params['date'] = pd.to_datetime(params['date'])
params = params.set_index('date').sort_index()

loadings_df = pd.read_parquet(_DATA / 'pca_loadings.parquet')
loadings_np = loadings_df.iloc[:3, 2:].to_numpy()  # (3,8): PC1/2/3 x [1,2,3,5,7,10,20,30]Y

CURRENT_HEDGE  = [2.0,  5.0, 10.0]
PROPOSED_HEDGE = [2.0, 10.0, 20.0]
param_cols = ['beta0', 'beta1', 'beta2', 'beta3', 'lambda1', 'lambda2']


def hedge_weights_for(sv, signal_mat, hedge_mats):
    sv_curve = np.array([svensson_zero_rate(t, *sv) for t in _FINE_GRID])
    hedge_coupons = [svensson_zero_rate(m, *sv) for m in hedge_mats]
    signal_coupon = svensson_zero_rate(signal_mat, *sv)
    res = build_hedge(
        signal_coupon, signal_mat,
        hedge_coupons, hedge_mats,
        _FINE_GRID, sv_curve, loadings_np,
    )
    return res['hedge_weights'], res['condition_number']


spot_dates_raw = [
    '1990-01-02', '1995-01-03', '2000-01-03', '2005-01-04',
    '2008-09-15', '2013-06-03', '2020-03-02', '2022-01-03', '2024-01-02',
]

for signal_mat, hedge_label in [(3.0, '3Y signal'), (7.0, '7Y signal')]:
    print(f'\n{hedge_label} -- proposed [2Y, 10Y, 20Y] hedge')
    print(f'  (weights in multiples of signal face; signal DV01 = $10,000)')
    print(f'  {"Date":<12} {"w_2Y":>8} {"w_10Y":>8} {"w_20Y":>8} {"PropCond":>9}  {"CurrCond":>9}')
    print(f'  {"-" * 62}')
    for d_raw in spot_dates_raw:
        try:
            d = params.index.asof(pd.Timestamp(d_raw))
            if pd.isna(d):
                print(f'  {d_raw:<12}  [no data before this date]')
                continue
            sv = params.loc[d, param_cols].to_numpy(float)
            if np.any(np.isnan(sv)):
                print(f'  {d_raw:<12}  [NaN Svensson params, skip]')
                continue
            w_new,  cond_new  = hedge_weights_for(sv, signal_mat, PROPOSED_HEDGE)
            _,      cond_curr = hedge_weights_for(sv, signal_mat, CURRENT_HEDGE)
            print(f'  {str(d.date()):<12} {w_new[0]:>8.3f} {w_new[1]:>8.3f} {w_new[2]:>8.3f} '
                  f'{cond_new:>9.1f}  {cond_curr:>9.1f}')
        except Exception as e:
            print(f'  {d_raw:<12}  [error: {e}]')

print('\nEconomic sanity: weights in [-5, +5] are normal for a 3-factor KRD hedge.')
print('Weights outside +/-10 would signal near-singular matrix or extreme curve shape.')


3Y signal -- proposed [2Y, 10Y, 20Y] hedge
  (weights in multiples of signal face; signal DV01 = $10,000)
  Date             w_2Y    w_10Y    w_20Y  PropCond   CurrCond
  --------------------------------------------------------------
  1990-01-02     -1.187   -0.153    0.041       8.1       95.5
  1995-01-03     -1.186   -0.152    0.041       8.1       95.8
  2000-01-03     -1.201   -0.144    0.036       8.2       98.9
  2005-01-04     -1.227   -0.130    0.031       8.5      105.8
  2008-09-15     -1.235   -0.127    0.029       8.7      107.8
  2013-06-03     -1.255   -0.117    0.025       9.2      114.7
  2020-03-02     -1.265   -0.104    0.019      10.9      122.0
  2022-01-03     -1.255   -0.108    0.020      10.4      119.2
  2024-01-02     -1.235   -0.121    0.026       9.2      109.0

7Y signal -- proposed [2Y, 10Y, 20Y] hedge
  (weights in multiples of signal face; signal DV01 = $10,000)
  Date             w_2Y    w_10Y    w_20Y  PropCond   CurrCond
  --------------------------

## Summary

### Part 1 — exit=0.0 results

See cell outputs for exact numbers. Key interpretation:

| Metric | Old (exit=2.0 stateless) | New (exit=0.0 hysteresis) |
|---|---|---|
| Gross Sharpe | 0.29 | *(see output)* |
| Net Sharpe   | 0.11 | *(see output)* |
| Net Total P&L | $313,804 | *(see output)* |
| Net Max Drawdown | -$647,353 | *(see output)* |

**What changes mechanically with exit=0.0:**
- Positions are held from entry at |z|>2.0 until z crosses zero — capturing the full mean-reversion
- Longer average holding period → fewer round trips → lower transaction costs
- Hit rate should improve (holding through noise to full reversion)
- Drawdown may increase (holding positions through overshoots)

---

### Part 2 — Hedge weight spot-check

**Weight sanity check interpretation:**
- Weights are in *multiples of signal_face* (signal face = notional sized so DV01=$10,000)
- Economically sane range: roughly -5 to +5 (i.e. hedge notional < 5× signal notional)
- Values outside ±10 would indicate near-singular conditions or extreme curve shapes
- The condition number for [2Y,10Y,20Y] should be ~4 vs ~54 for [2Y,5Y,10Y] (from notebook 34)

**20Y Treasury liquidity caveat:**

The 20Y Treasury was *discontinued in 1986* and only *reintroduced in May 2020*.
- **Pre-2020 (1971–2020):** any '20Y' hedge would use off-the-run bonds or synthetic
  construction from 10Y/30Y, not a clean benchmark bond. Bid-ask spreads for off-the-run
  20Y: approximately 1–3 bp, versus 0.25 bp for benchmark 2Y/5Y/10Y.
- **Post-2020:** the 20Y is on-the-run but thinner than 10Y. Typical bid-ask ~0.75–1.5 bp.
- **Backtest cost model:** charges flat 0.02 bp/leg regardless of maturity — this
  *understates 20Y execution cost by 10–100× for the pre-2020 period.*
- **Recommendation:** keep [2Y, 5Y, 10Y] as the official hedge set for backtest credibility.
  Use the condition-number result from notebook 34 as evidence the hedge is not ill-conditioned —
  not as a live-trading recommendation to switch instruments.